# 🍽️ Smart Canteen System - Exploratory Data Analysis (EDA)

### Objectives:
1. Understand order volumes, peak hours, and customer dining patterns.
2. Identify top revenue-generating menu items and categories.
3. Analyze customer waiting times and satisfaction ratings.
4. Uncover the impact of exam weeks and weekends on campus food demand.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11


## 1. Load and Inspect Canteen Dataset


In [ ]:
data_path = os.path.join('..', 'data', 'canteen_orders.csv')
df = pd.read_csv(data_path)
df['date'] = pd.to_datetime(df['date'])
print(f'Total Records: {len(df):,}')
print(f'Total Unique Orders: {df["order_id"].nunique():,}')
df.head()


In [ ]:
# Check data types and missing values
df.info()
print('\nMissing values count:')
print(df.isnull().sum())


In [ ]:
# Summary statistics for numerical columns
df[['unit_price', 'quantity', 'total_price', 'wait_time_min', 'rating']].describe().round(2)


## 2. Order Volume by Hour & Peak Rush Analysis
Identifying rush hours helps kitchen management allocate staff and prepare ingredients beforehand.


In [ ]:
plt.figure(figsize=(10, 4))
hourly_orders = df.groupby('hour')['order_id'].nunique().reset_index()
ax = sns.barplot(data=hourly_orders, x='hour', y='order_id', palette='Blues_d')
plt.title('Order Distribution by Hour of the Day', fontsize=14, fontweight='bold')
plt.xlabel('Hour of Day (24-Hour Format)')
plt.ylabel('Unique Orders Placed')
plt.show()


**Observation:**
- Lunch hours (**12:00 PM – 2:00 PM**) and evening snacks (**4:00 PM – 6:00 PM**) experience the heaviest footfall.
- Additional serving counters should be staffed during these windows.


## 3. Menu Item Performance & Revenue Share


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sales by category
cat_summary = df.groupby('category')['total_price'].sum().sort_values(ascending=False)
cat_summary.plot(kind='pie', autopct='%1.1f%%', ax=axes[0], colors=['#34d399', '#60a5fa', '#f472b6', '#fbbf24'])
axes[0].set_title('Revenue Share by Category', fontweight='bold')
axes[0].set_ylabel('')

# Top 8 menu items
top_items = df.groupby('item_name')['quantity'].sum().sort_values(ascending=False).head(8)
sns.barplot(x=top_items.values, y=top_items.index, ax=axes[1], palette='crest')
axes[1].set_title('Top 8 Items by Units Sold', fontweight='bold')
axes[1].set_xlabel('Total Units Sold')

plt.tight_layout()
plt.show()


## 4. Impact of Exam Weeks & Weekends on Canteen Demand


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Exam vs Regular Week Average Daily Orders
daily_exam = df.groupby(['date', 'is_exam_week'])['order_id'].nunique().reset_index()
sns.boxplot(data=daily_exam, x='is_exam_week', y='order_id', ax=axes[0], palette=['#93c5fd', '#fca5a5'])
axes[0].set_xticklabels(['Regular Week', 'Exam Week'])
axes[0].set_title('Daily Orders: Regular vs Exam Weeks', fontweight='bold')
axes[0].set_ylabel('Daily Orders')

# Category shift during exams
exam_cat = df.groupby(['is_exam_week', 'category'])['quantity'].sum().unstack()
exam_cat_pct = exam_cat.div(exam_cat.sum(axis=1), axis=0) * 100
exam_cat_pct.plot(kind='bar', stacked=True, ax=axes[1], colormap='Spectral')
axes[1].set_xticklabels(['Regular Week', 'Exam Week'], rotation=0)
axes[1].set_title('Category Proportion Shift during Exams', fontweight='bold')
axes[1].set_ylabel('Percentage of Items Sold (%)')
axes[1].legend(title='Category', bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.show()


## 5. Wait Time vs Customer Ratings


In [ ]:
plt.figure(figsize=(9, 4.5))
sns.boxplot(data=df, x='rating', y='wait_time_min', palette='RdYlGn')
plt.title('Impact of Wait Time on Customer Satisfaction Rating', fontsize=13, fontweight='bold')
plt.xlabel('Customer Rating (1 to 5 Stars)')
plt.ylabel('Wait Time (Minutes)')
plt.show()


## Summary of Findings:
1. **Peak Demand**: Peak rushes occur between 13:00 - 15:00 (Lunch) and 16:00 - 18:00 (Evening tea & snacks).
2. **Category Driver**: Meals and Beverages account for over 65% of total canteen revenue.
3. **Exam Period Behavior**: During exam weeks, students consume significantly more Beverages (Cold Coffee, Chai) and quick Snacks rather than full heavy meals.
4. **Service Speed**: Orders taking longer than 8-10 minutes correlate directly with lower (1-2 star) ratings, indicating that kitchen prep efficiency is essential.
